# 02 — Preparación de datos (CRISP-DM: fase 3)

Construimos la misma tabla que el nodo Kedro `build_ml_features_table`: cuotas + `outcome` + goles.

**Prerrequisito:** haber visto `01_...`. **Siguiente:** `03_clasificacion_resultado.ipynb`.

In [ ]:
%load_ext kedro.ipython

In [ ]:
import numpy as np
import pandas as pd

match = catalog.load("match")

goal_diff = match["home_team_goal"] - match["away_team_goal"]
outcome = np.select(
    [goal_diff > 0, goal_diff == 0, goal_diff < 0],
    [2, 1, 0],
)
feat_cols = ["B365H", "B365D", "B365A"]
cols = feat_cols + ["home_team_goal", "away_team_goal"]
df = match[cols].copy()
df["outcome"] = outcome
df_ml = df.dropna(subset=feat_cols).reset_index(drop=True)

print(f"Filas tras quitar NaN en B365: {len(df_ml):,} (antes {len(match):,})")
df_ml.head()

In [ ]:
# Distribución del outcome (desbalance entre clases)
df_ml["outcome"].value_counts(normalize=True).sort_index()

In [ ]:
# Comparar con salida de Kedro si ya ejecutaste `kedro run`
from pathlib import Path

path = Path("data/05_model_input/features_for_ml.parquet")
if path.is_file():
    kedro_df = pd.read_parquet(path)
    assert len(kedro_df) == len(df_ml), (len(kedro_df), len(df_ml))
    assert set(kedro_df.columns) == set(df_ml.columns)
    # Smoke test: mismas sumas por columna numérica
    for c in feat_cols + ["home_team_goal", "away_team_goal", "outcome"]:
        assert float(df_ml[c].sum()) == float(kedro_df[c].sum())
    print("OK: tabla notebook alineada con Parquet de Kedro (filas, columnas y sumas).")
else:
    print("Aún no existe Parquet; ejecuta `kedro run --pipeline data_processing` desde la raíz del proyecto.")